In [1]:
from tifffile import imread
import os
from matplotlib import pyplot as plt
import json
import geojson
from scipy.io import loadmat
import h5py
import numpy as np
from functions import *

In [2]:
# pth_WSI_segmentations = r'\\10.99.68.178\andreex\data\Stardist\PDAC model\maybe_all_tiles_donald\test_WSI\StarDist_2_9_24\json'
#pth_WSI_segmentations = r'\\10.99.68.178\andre\Eduarda PDAC\immune_cell_pipeline\star_dist_Donald_model\StarDist_1_29_23\json'

# pth_WSI_segmentations = r'\\10.162.80.16\Andre_expansion\data\monkey_fetus\Stardist\StarDist_12_25_23\json'
npdi_im_path = r'\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55'
pth_WSI_segmentations = os.path.join(npdi_im_path,'StarDist_12_11_23', 'json')
out_pth = os.path.join(pth_WSI_segmentations, 'downsampled_jsons', '32_polys_20x')
os.makedirs(out_pth, exist_ok=True)

# Get list of paths
json_pth_list = get_sorted_files(pth_WSI_segmentations, '.json')

print("npdi_im_path:", npdi_im_path)
print("pth_WSI_segmentations:", pth_WSI_segmentations)
print("out_pth:", out_pth)
# ds_amt = 4 # 2.5x
# ds_amt = 2/0.4416  # 5x
# ds_amt = 1/0.4416 # 10x
ds_amt = 1 # 20x


npdi_im_path: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55
pth_WSI_segmentations: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_11_23\json
out_pth: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_11_23\json\downsampled_jsons\32_polys_20x


In [3]:
num_slide=49
print([json_pth_list[num_slide]])

['\\\\10.162.80.16\\Andre\\data\\monkey fetus\\bissected_monkey_GS55\\StarDist_12_11_23\\json\\monkey_fetus_GS55_0148.json']


In [24]:
# for p, file in enumerate(json_pth_list):
for p, file in enumerate([json_pth_list[num_slide]]):
    nm = file.split('\\')[-1]
    new_fn = os.path.join(out_pth, nm[:-5] + '.geojson')
    print(f'{p} / {len(json_pth_list)}')
    print(nm)
    
    if not os.path.exists(new_fn):
    
        segmentation_data = json.load(open(file))
        
        data_list = get_ds_data(segmentation_data, ds_amt)
    
        GEOdata = []
        
        for j, (centroid, contour) in enumerate(data_list):
            
            if j % 1000 == 0:
                print(f'Processing contour {j+1}/{len(data_list)}')
            
            centroid = [centroid[0] + 0, centroid[1] + 0]
            # xy coordinates are swapped, so I reverse them here with xy[::-1]
            # note: add 1 to coords to fix 0 indexing vs 1 index offset
            contour = [[coord+0 for coord in xy[::-1]] for xy in contour]  # Convert coordinates to integers
            contour.append(contour[0]) # stardist doesn't close the circle, needed for qupath
        
            # Create a new dictionary for each contour
            dict_data = {
                "type": "Feature",
                "id": "PathCellObject",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [contour]
                },
                "properties": {
                    'objectType': 'annotation',
                    'classification': {'name': 'Nuclei', 'color': [97, 214, 59]}
                }
            }
        
            GEOdata.append(dict_data)
        
        with open(new_fn,'w') as outfile:
            geojson.dump(GEOdata,outfile)
        print('Finished',new_fn)
    
    else:
        print(f'skipping {new_fn}')

0 / 116
monkey_fetus_GS55_0148.json
Finished \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\downsampled_jsons\32_polys_20x\monkey_fetus_GS55_0148.geojson
